# GRPO Math Reasoning with alignrl

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sacredvoid/alignrl/blob/main/notebooks/02_grpo_math_reasoning.ipynb)

This notebook demonstrates **Group Relative Policy Optimization (GRPO)**, a reinforcement learning technique that teaches models to reason about math problems. Unlike SFT where we show the model correct answers, GRPO lets the model explore different solution strategies and reinforces the ones that produce correct, well-formatted answers. The key insight: we use *verifiable rewards* (checking if the answer is actually right) instead of a learned reward model.

## Setup

Install alignrl with training and Unsloth dependencies.

In [ ]:
!pip install alignrl[train,unsloth] -q

## How GRPO Works

GRPO (from DeepSeek-R1) is a policy optimization algorithm that avoids the need for a separate critic/value model. Here's the core idea:

1. **Sample a group**: For each math problem, generate `G` different completions (e.g., G=4)
2. **Score each completion**: Use reward functions to assign scores (correct answer? good format?)
3. **Compute relative advantage**: Within the group, subtract the mean reward and divide by the std. Completions better than the group average get positive advantage; worse ones get negative.
4. **Policy gradient**: Update the model to increase the probability of high-advantage completions and decrease low-advantage ones.

```
Problem: "What is 15% of 240?"
          |
          v
    +--Generate G=4 completions--+
    |          |          |          |
    v          v          v          v
  Comp 1     Comp 2     Comp 3     Comp 4
  \boxed{36} \boxed{38} \boxed{36} (no box)
  R=2.0      R=0.0      R=2.0      R=0.0
    |          |          |          |
    v          v          v          v
  Advantage = (R - mean(R)) / std(R)
  +1.0       -1.0       +1.0       -1.0
    |          |          |          |
    v          v          v          v
  Increase   Decrease   Increase   Decrease
  prob       prob       prob       prob
```

### Reward Functions

We use two reward functions that stack:
- **`math_verify_reward`**: Binary (0/1) - does the extracted answer match the ground truth?
- **`format_reward`**: 0.0/0.5/1.0 - does the response use `\boxed{}` format correctly?

## Configuration

Key GRPO parameters:
- **num_generations=4**: Generate 4 completions per prompt (the "group" in GRPO)
- **beta=0.001**: KL penalty weight (keeps the model from drifting too far from the base)
- **learning_rate=5e-6**: Lower than SFT since RL training is more sensitive

In [ ]:
from alignrl.grpo import GRPOConfig, GRPORunner

config = GRPOConfig(
    model_name="Qwen/Qwen2.5-3B",
    output_dir="./outputs/grpo",
    dataset_name="openai/gsm8k",
    dataset_config="main",
    dataset_size=500,
    max_steps=100,
    num_generations=4,
    max_completion_length=512,
    max_prompt_length=256,
    beta=0.001,
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    lora_r=16,
    lora_alpha=32,
    load_in_4bit=True,
    report_to="none",
    logging_steps=5,
)

print(config.model_dump_json(indent=2))

## Data Exploration

GSM8K contains grade-school math word problems with step-by-step solutions. The final answer is marked with `####`.

In [ ]:
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main", split="train")
print(f"Full dataset size: {len(ds):,} examples")
print(f"Columns: {ds.column_names}")
print()

# Show a few examples
for i in range(3):
    ex = ds[i]
    answer = ex["answer"].split("####")[-1].strip()
    print(f"--- Example {i+1} ---")
    print(f"Question: {ex['question'][:200]}")
    print(f"Answer: {answer}")
    print()

### Reward Functions Deep Dive

Let's see how the reward functions work on some example completions:

In [ ]:
from alignrl.rewards import math_verify_reward, format_reward, extract_answer

# Simulate some model completions
test_completions = [
    [{"content": "15% of 240 = 0.15 * 240 = 36. The answer is \\boxed{36}"}],
    [{"content": "Let me calculate: 240 * 15 / 100 = 38"}],
    [{"content": "\\boxed{36}\n\nSo the answer is 36."}],
    [{"content": "I think it might be around 40 or so."}],
]
solutions = ["36", "36", "36", "36"]

math_rewards = math_verify_reward(test_completions, solutions)
fmt_rewards = format_reward(test_completions)

print(f"{'Completion (truncated)':<55} {'Math':>5} {'Fmt':>5} {'Total':>6}")
print("-" * 75)
for comp, mr, fr in zip(test_completions, math_rewards, fmt_rewards):
    text = comp[0]['content'][:50] + ("..." if len(comp[0]['content']) > 50 else "")
    print(f"{text:<55} {mr:>5.1f} {fr:>5.1f} {mr+fr:>6.1f}")

## Training

The `GRPORunner` handles model loading, dataset formatting, and the GRPO training loop. This will take longer than SFT because each step requires generating multiple completions per prompt.

In [ ]:
runner = GRPORunner(config)
result = runner.train()

print(f"Training complete!")
print(f"  Steps: {result.num_steps}")
print(f"  Final loss: {result.loss_history[-1]:.4f}")
print(f"  Adapter saved to: {result.output_dir}")

## Results: Reward and Loss Curves

The most important plot for GRPO is the **reward curve**: are the model's answers getting more correct over time?

In [ ]:
import json
import matplotlib.pyplot as plt

# Load the full training log
with open("./outputs/grpo/train_result.json") as f:
    train_log = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(train_log["loss_history"], linewidth=2, color="#dc2626")
axes[0].set_xlabel("Logging Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("GRPO Training Loss")
axes[0].grid(True, alpha=0.3)

# Reward curve
if train_log.get("reward_history"):
    axes[1].plot(train_log["reward_history"], linewidth=2, color="#16a34a")
    axes[1].set_xlabel("Logging Step")
    axes[1].set_ylabel("Mean Reward")
    axes[1].set_title("GRPO Reward Over Training")
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "No reward data logged", ha="center", va="center")

plt.tight_layout()
plt.show()

## Inference: Example Reasoning Chains

Let's see the GRPO-trained model solve math problems. The model should produce step-by-step reasoning with the final answer in `\boxed{}`.

In [ ]:
from alignrl.inference import InferenceConfig, ModelServer, build_prompt

MATH_SYSTEM = (
    "You are a helpful math assistant. Solve problems step by step, "
    "showing your reasoning clearly. Put your final answer in \\boxed{}."
)

server = ModelServer(InferenceConfig(
    model_name="Qwen/Qwen2.5-3B",
    adapter_path="./outputs/grpo/final",
    backend="unsloth",
    max_tokens=512,
    temperature=0.7,
))
server.load()

math_problems = [
    "What is 15% of 240?",
    "A store sells apples for $2 each. If you buy 5 or more, you get a 20% discount. How much do 7 apples cost?",
    "If a train travels at 60 mph for 2.5 hours, how far does it go?",
    "Sally has 3 times as many marbles as Tom. Together they have 48 marbles. How many does Sally have?",
]

for problem in math_problems:
    messages = build_prompt(problem, system=MATH_SYSTEM)
    response = server.generate(messages)
    print(f"Q: {problem}")
    print(f"A: {response}")
    print("=" * 60)

## Save Adapter Weights and Results

In [ ]:
from pathlib import Path

output_dir = Path("./outputs/grpo")

print("Saved files:")
for f in sorted(output_dir.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  {f.relative_to(output_dir)} ({size_mb:.1f} MB)")

print(f"\nGRPO adapter ready at: {output_dir / 'final'}")
print("Use this adapter as the starting point for DPO (Notebook 03) or evaluation (Notebook 04).")

## Next Steps

The GRPO-trained adapter has improved the model's math reasoning ability through reinforcement learning with verifiable rewards. In the next notebooks:

- **Notebook 03 (DPO)**: Align the model with human preferences for response quality
- **Notebook 04 (Evaluation)**: Benchmark all training stages on standard benchmarks
- **Notebook 05 (Inference)**: Serve the trained model with optimized backends